# M2 形式のファイルを読み書きする

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/17_gec_education/m2_format.ipynb)

NLP クックブック 17-04 ／ 文法誤り訂正と言語教育

CoNLL-2014 や BEA-2019 などの GEC データセットで使われる M2 形式を読み込み、訂正文を復元したい。

In [1]:
m2 = """S I has went to school yesterday and buyed a apple .
A 1 2|||U:VERB:TENSE||||||REQUIRED|||-NONE-|||0
A 7 8|||R:VERB:INFL|||bought|||REQUIRED|||-NONE-|||0
A 8 9|||R:DET|||an|||REQUIRED|||-NONE-|||0
A 1 3|||R:VERB:TENSE|||went|||REQUIRED|||-NONE-|||1
A 7 8|||R:VERB:INFL|||bought|||REQUIRED|||-NONE-|||1
A 8 9|||R:DET|||an|||REQUIRED|||-NONE-|||1

S This sentence is correct .
A -1 -1|||noop|||-NONE-|||REQUIRED|||-NONE-|||0
"""

def read_m2(text):
    """[(原文のトークン列, {アノテータ ID: [(開始, 終了, タイプ, 訂正), ...]}), ...]"""
    data = []
    for block in text.strip().split("\n\n"):
        lines = block.split("\n")
        tokens = lines[0][2:].split()
        edits = {}
        for line in lines[1:]:
            span, typ, corr, _, _, annotator = line[2:].split("|||")
            start, end = map(int, span.split())
            edits.setdefault(int(annotator), [])
            if typ != "noop":
                edits[int(annotator)].append((start, end, typ, corr))
        data.append((tokens, edits))
    return data

def apply_edits(tokens, edits):
    out = list(tokens)
    for start, end, _, corr in sorted(edits, reverse=True):     # 後ろから適用すれば位置がずれない
        out[start:end] = corr.split() if corr and corr != "-NONE-" else []
    return " ".join(out)

for tokens, edits in read_m2(m2):
    print("原文:", " ".join(tokens))
    for annotator, es in edits.items():
        print(f"  アノテータ {annotator}: {apply_edits(tokens, es)}   {[(s, e, t) for s, e, t, _ in es]}")

原文: I has went to school yesterday and buyed a apple .
  アノテータ 0: I went to school yesterday and bought an apple .   [(1, 2, 'U:VERB:TENSE'), (7, 8, 'R:VERB:INFL'), (8, 9, 'R:DET')]
  アノテータ 1: I went to school yesterday and bought an apple .   [(1, 3, 'R:VERB:TENSE'), (7, 8, 'R:VERB:INFL'), (8, 9, 'R:DET')]
原文: This sentence is correct .
  アノテータ 0: This sentence is correct .   []


- `S` 行が原文、`A` 行が 1 つの編集 (`開始 終了|||タイプ|||訂正|||必須か|||コメント|||アノテータ ID`)。位置はトークン単位で、終了は含まない。
- 訂正が不要な文は `noop` の行が 1 つ付く。アノテータが複数いる場合は、同じ文に対して ID の違う編集の集合が並ぶ。